# NVIDIA Nemotron Model Reasoning Challenge
## Part 1: Data Analysis & Programmatic Solvers

**Goal**: Understand the dataset, build programmatic solvers for all 6 puzzle categories, and generate Chain-of-Thought (CoT) training data.

**Competition**: Produce a LoRA adapter (rank ≤ 32) for Nemotron-3-Nano-30B that maximizes accuracy on reasoning puzzles.

In [ ]:
import csv
import re
import numpy as np
import pandas as pd
from collections import Counter, defaultdict

# Load data
train_df = pd.read_csv('./train.csv')
test_df = pd.read_csv('./test.csv')

print(f'Training set: {len(train_df)} puzzles')
print(f'Test set: {len(test_df)} puzzles')
print(f'\nColumns: {list(train_df.columns)}')
train_df.head()

Training set: 9500 puzzles
Test set: 3 puzzles

Columns: ['id', 'prompt', 'answer']


,id,prompt,answer
0,00066667,"In Alice's Wonderland, a secret bit manipulati...",10010111
1,000b53cf,"In Alice's Wonderland, a secret bit manipulati...",01000011
2,00189f6a,"In Alice's Wonderland, secret encryption rules...",cat imagines book
3,001b24c4,"In Alice's Wonderland, numbers are secretly co...",XXXVIII
4,001c63cb,"In Alice's Wonderland, secret encryption rules...",wizard creates secret


## 1. Categorize Puzzles

In [2]:
def classify_puzzle(prompt):
    """Classify a puzzle by its category."""
    if 'bit manipulation' in prompt:
        return 'bit_manipulation'
    elif 'encryption' in prompt:
        return 'text_encryption'
    elif 'numeral system' in prompt:
        return 'numeral_system'
    elif 'unit conversion' in prompt:
        return 'unit_conversion'
    elif 'gravitational constant' in prompt:
        return 'gravitational'
    elif 'transformation rules' in prompt:
        return 'transformation_rules'
    return 'unknown'

train_df['category'] = train_df['prompt'].apply(classify_puzzle)
test_df['category'] = test_df['prompt'].apply(classify_puzzle)

print('=== Training Set Category Distribution ===')
print(train_df['category'].value_counts())
print(f'\n=== Test Set Category Distribution ===')
print(test_df['category'].value_counts())

=== Training Set Category Distribution ===
category
bit_manipulation        1602
gravitational           1597
unit_conversion         1594
text_encryption         1576
numeral_system          1576
transformation_rules    1555
Name: count, dtype: int64

=== Test Set Category Distribution ===
category
bit_manipulation    2
text_encryption     1
Name: count, dtype: int64


In [3]:
# Prompt length analysis
train_df['prompt_len'] = train_df['prompt'].str.len()
train_df['answer_len'] = train_df['answer'].astype(str).str.len()

print('=== Prompt Length by Category ===')
print(train_df.groupby('category')['prompt_len'].describe().round(0))
print('\n=== Answer Length by Category ===')
print(train_df.groupby('category')['answer_len'].describe().round(0))

=== Prompt Length by Category ===
                       count   mean   std    min    25%    50%    75%    max
category                                                                    
bit_manipulation      1602.0  479.0  23.0  447.0  468.0  489.0  489.0  510.0
gravitational         1597.0  320.0  28.0  280.0  286.0  319.0  352.0  357.0
numeral_system        1576.0  219.0  10.0  199.0  210.0  219.0  228.0  241.0
text_encryption       1576.0  370.0  50.0  265.0  328.0  370.0  412.0  485.0
transformation_rules  1555.0  195.0  10.0  177.0  184.0  195.0  205.0  212.0
unit_conversion       1594.0  222.0  18.0  197.0  202.0  223.0  244.0  246.0

=== Answer Length by Category ===
                       count  mean  std   min   25%   50%   75%   max
category                                                             
bit_manipulation      1602.0   8.0  0.0   8.0   8.0   8.0   8.0   8.0
gravitational         1597.0   5.0  1.0   3.0   5.0   5.0   5.0   6.0
numeral_system        1576.0   4.0 

In [4]:
# Show example from each category
for cat in train_df['category'].unique():
    row = train_df[train_df['category'] == cat].iloc[0]
    print(f'\n{"="*70}')
    print(f'Category: {cat}')
    print(f'Answer: {row["answer"]}')
    print(f'Prompt (first 400 chars):')
    print(row['prompt'][:400])
    print('...')


Category: bit_manipulation
Answer: 10010111
Prompt (first 400 chars):
In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.

Here are some examples of input -> output:
01010001 -> 11011101
00001001 -> 01101101
00010101 -> 01010101
11111111 -> 10000001
10011101 -> 01000101
00111011 -> 00001001
10111101 -> 00
...

Category: text_encryption
Answer: cat imagines book
Prompt (first 400 chars):
In Alice's Wonderland, secret encryption rules are used on text. Here are some examples:
ucoov pwgtfyoqg vorq yrjjoe -> queen discovers near valley
pqrsfv pqorzg wvgwpo trgbjo -> dragon dreams inside castle
gbcpovb tqorbog bxo zrswtrj pffq -> student creates the magical door
bxo sfjpov pqrsfv dfjjfig -> the golden dragon follows
nqwvtogg qorpg bxo zegboqwfcg gotqob -> princess reads the mysterious
...

Category: numeral_system
Answer: XXXVII

## 2. Solvers

### 2.1 Numeral System (Decimal → Roman Numerals) — Expected: ~100%

In [5]:
def int_to_roman(num):
    val = [1000, 900, 500, 400, 100, 90, 50, 40, 10, 9, 5, 4, 1]
    syms = ['M', 'CM', 'D', 'CD', 'C', 'XC', 'L', 'XL', 'X', 'IX', 'V', 'IV', 'I']
    result = ''
    for i in range(len(val)):
        while num >= val[i]:
            result += syms[i]
            num -= val[i]
    return result

def solve_numeral_system(prompt):
    match = re.search(r'write the number (\d+)', prompt)
    if not match:
        return None
    return int_to_roman(int(match.group(1)))

# Validate
subset = train_df[train_df['category'] == 'numeral_system'].copy()
subset['predicted'] = subset['prompt'].apply(solve_numeral_system)
subset['correct'] = subset['predicted'] == subset['answer'].astype(str)
print(f"Numeral System: {subset['correct'].sum()}/{len(subset)} ({subset['correct'].mean()*100:.1f}%)")

Numeral System: 1576/1576 (100.0%)


### 2.2 Gravitational Constant (d = 0.5 * g * t²)

In [6]:
def solve_gravitational(prompt):
    """Extract g via least-squares fit, compute distance for query time."""
    examples = re.findall(r'For t = ([\d.]+)s, distance = ([\d.]+) m', prompt)
    query_match = re.search(r'for t = ([\d.]+)s', prompt)
    if not examples or not query_match:
        return None

    ts = np.array([float(t) for t, d in examples])
    ds = np.array([float(d) for t, d in examples])
    t_query = float(query_match.group(1))

    # Least squares: minimize sum((d_i - 0.5*g*t_i^2)^2)
    # g = 2 * sum(d_i * t_i^2) / sum(t_i^4)
    g_lsq = 2 * np.sum(ds * ts**2) / np.sum(ts**4)
    result = 0.5 * g_lsq * t_query**2
    return f"{result:.2f}"

# Validate
subset = train_df[train_df['category'] == 'gravitational'].copy()
subset['predicted'] = subset['prompt'].apply(solve_gravitational)
subset['correct'] = subset['predicted'] == subset['answer'].astype(str)
acc = subset['correct'].mean()
print(f"Gravitational: {subset['correct'].sum()}/{len(subset)} ({acc*100:.1f}%)")

# Check how close the wrong ones are
wrong = subset[~subset['correct']].copy()
wrong['expected_num'] = wrong['answer'].astype(float)
wrong['predicted_num'] = wrong['predicted'].astype(float)
wrong['abs_error'] = (wrong['expected_num'] - wrong['predicted_num']).abs()
wrong['rel_error'] = wrong['abs_error'] / wrong['expected_num'].abs()
print(f"\nWrong predictions error stats:")
print(f"  Max absolute error: {wrong['abs_error'].max():.4f}")
print(f"  Mean absolute error: {wrong['abs_error'].mean():.4f}")
print(f"  Max relative error: {wrong['rel_error'].max():.6f}")
print(f"  Mean relative error: {wrong['rel_error'].mean():.6f}")
print(f"  All within 0.01 rel tolerance: {(wrong['rel_error'] < 0.01).all()}")

Gravitational: 1279/1597 (80.1%)

Wrong predictions error stats:
  Max absolute error: 0.0200
  Mean absolute error: 0.0057
  Max relative error: 0.001658
  Mean relative error: 0.000102
  All within 0.01 rel tolerance: True


### 2.3 Unit Conversion (Linear Scaling)

In [7]:
def solve_unit_conversion(prompt):
    """Extract scaling factor from examples, apply to query."""
    examples = re.findall(r'([\d.]+) m becomes ([\d.]+)', prompt)
    query_match = re.search(r'convert the following measurement: ([\d.]+) m', prompt)
    if not examples or not query_match:
        return None

    xs = np.array([float(x) for x, y in examples])
    ys = np.array([float(y) for x, y in examples])
    x_query = float(query_match.group(1))

    # Least-squares fit: y = factor * x
    # factor = sum(x*y) / sum(x^2)
    factor = np.sum(xs * ys) / np.sum(xs**2)
    result = factor * x_query
    return f"{result:.2f}"

# Validate
subset = train_df[train_df['category'] == 'unit_conversion'].copy()
subset['predicted'] = subset['prompt'].apply(solve_unit_conversion)
subset['correct'] = subset['predicted'] == subset['answer'].astype(str)
acc = subset['correct'].mean()
print(f"Unit Conversion: {subset['correct'].sum()}/{len(subset)} ({acc*100:.1f}%)")

wrong = subset[~subset['correct']].copy()
if len(wrong) > 0:
    wrong['expected_num'] = wrong['answer'].astype(float)
    wrong['predicted_num'] = wrong['predicted'].astype(float)
    wrong['abs_error'] = (wrong['expected_num'] - wrong['predicted_num']).abs()
    wrong['rel_error'] = wrong['abs_error'] / wrong['expected_num'].abs()
    print(f"Wrong predictions error stats:")
    print(f"  Max absolute error: {wrong['abs_error'].max():.4f}")
    print(f"  Max relative error: {wrong['rel_error'].max():.6f}")

Unit Conversion: 1427/1594 (89.5%)
Wrong predictions error stats:
  Max absolute error: 0.0100
  Max relative error: 0.001942


### 2.4 Text Encryption (Substitution Cipher)

In [8]:
def solve_text_encryption(prompt):
    """Solve substitution cipher by building char mapping from examples."""
    lines = prompt.strip().split('\n')

    examples = []
    query_text = None

    for line in lines:
        line = line.strip()
        if ' -> ' in line and 'encryption' not in line.lower():
            parts = line.split(' -> ')
            if len(parts) == 2:
                examples.append((parts[0].strip(), parts[1].strip()))
        elif 'decrypt the following text:' in line.lower():
            match = re.search(r'decrypt the following text:\s*(.*)', line, re.IGNORECASE)
            if match:
                query_text = match.group(1).strip()

    if not examples or not query_text:
        return None

    # Build character mapping
    char_map = {}
    conflicts = set()
    for encrypted, decrypted in examples:
        if len(encrypted) != len(decrypted):
            continue
        for e, d in zip(encrypted, decrypted):
            if e == ' ' and d == ' ':
                continue
            if e in char_map:
                if char_map[e] != d:
                    conflicts.add(e)
            else:
                char_map[e] = d

    # Remove conflicting mappings
    for c in conflicts:
        del char_map[c]

    result = []
    unmapped = 0
    for c in query_text:
        if c == ' ':
            result.append(' ')
        elif c in char_map:
            result.append(char_map[c])
        else:
            result.append('?')
            unmapped += 1

    return ''.join(result)

# Validate
subset = train_df[train_df['category'] == 'text_encryption'].copy()
subset['predicted'] = subset['prompt'].apply(solve_text_encryption)
subset['correct'] = subset['predicted'] == subset['answer'].astype(str)
acc = subset['correct'].mean()
print(f"Text Encryption: {subset['correct'].sum()}/{len(subset)} ({acc*100:.1f}%)")

# Show some failures
wrong = subset[~subset['correct']].head(5)
for _, row in wrong.iterrows():
    print(f"  Expected: {row['answer']}, Got: {row['predicted']}")

Text Encryption: 605/1576 (38.4%)
  Expected: cat imagines book, Got: cat imagines ?oo?
  Expected: king chases castle, Got: ?ing chases castle
  Expected: alice watches under wonderland, Got: alice ?atches under ?onderland
  Expected: wizard watches through library, Got: wi?ard watches through library
  Expected: turtle watches the mysterious garden, Got: turtle watches the m?sterious garden


In [9]:
# Deeper analysis of text encryption: are these always simple substitution ciphers?
# Let's check if the mapping is consistent (each encrypted char maps to exactly one decrypted char)

def analyze_cipher(prompt):
    lines = prompt.strip().split('\n')
    examples = []
    for line in lines:
        line = line.strip()
        if ' -> ' in line and 'encryption' not in line.lower():
            parts = line.split(' -> ')
            if len(parts) == 2:
                examples.append((parts[0].strip(), parts[1].strip()))
    
    # Check if all mappings are length-preserving
    length_preserving = all(len(e) == len(d) for e, d in examples)
    
    # Build forward and reverse maps
    fwd_map = defaultdict(set)  # encrypted char -> set of decrypted chars
    rev_map = defaultdict(set)  # decrypted char -> set of encrypted chars
    for enc, dec in examples:
        if len(enc) == len(dec):
            for e, d in zip(enc, dec):
                if e != ' ':
                    fwd_map[e].add(d)
                    rev_map[d].add(e)
    
    # Check if it's a clean 1-to-1 mapping
    fwd_conflicts = sum(1 for v in fwd_map.values() if len(v) > 1)
    rev_conflicts = sum(1 for v in rev_map.values() if len(v) > 1)
    
    return {
        'length_preserving': length_preserving,
        'fwd_conflicts': fwd_conflicts,
        'rev_conflicts': rev_conflicts,
        'mapped_chars': len(fwd_map),
        'n_examples': len(examples)
    }

enc_subset = train_df[train_df['category'] == 'text_encryption'].copy()
analyses = enc_subset['prompt'].apply(analyze_cipher).apply(pd.Series)

print('Text Encryption Analysis:')
print(f"All length-preserving: {analyses['length_preserving'].all()}")
print(f"Puzzles with forward conflicts: {(analyses['fwd_conflicts'] > 0).sum()}/{len(analyses)}")
print(f"Puzzles with reverse conflicts: {(analyses['rev_conflicts'] > 0).sum()}/{len(analyses)}")
print(f"\nMapped chars per puzzle: mean={analyses['mapped_chars'].mean():.1f}, min={analyses['mapped_chars'].min()}, max={analyses['mapped_chars'].max()}")
print(f"Examples per puzzle: mean={analyses['n_examples'].mean():.1f}, min={analyses['n_examples'].min()}, max={analyses['n_examples'].max()}")

Text Encryption Analysis:
All length-preserving: True
Puzzles with forward conflicts: 0/1576
Puzzles with reverse conflicts: 0/1576

Mapped chars per puzzle: mean=20.0, min=14, max=25
Examples per puzzle: mean=4.0, min=3, max=5


### 2.5 Bit Manipulation

In [10]:
def solve_bit_manipulation(prompt):
    """
    Try common bit manipulation operations to find the rule.
    Strategy: try single ops first, then 2-op compositions.
    """
    examples = re.findall(r'([01]{8}) -> ([01]{8})', prompt)
    query_match = re.search(r'determine the output for:\s*([01]{8})', prompt)
    if not examples or not query_match:
        return None

    query = query_match.group(1)

    def b2i(s): return int(s, 2)
    def i2b(n): return format(n & 0xFF, '08b')

    # Build list of atomic operations
    ops = []
    
    # Rotations
    for n in range(1, 8):
        ops.append(('rot_l_' + str(n), lambda s, n=n: s[n:] + s[:n]))
        ops.append(('rot_r_' + str(n), lambda s, n=n: s[8-n:] + s[:8-n]))
    
    # Reverse
    ops.append(('reverse', lambda s: s[::-1]))
    
    # NOT
    ops.append(('not', lambda s: i2b(b2i(s) ^ 0xFF)))
    
    # XOR with constants
    for c in range(1, 256):
        ops.append(('xor_' + str(c), lambda s, c=c: i2b(b2i(s) ^ c)))
    
    # Swap nibbles
    ops.append(('swap_nibbles', lambda s: s[4:] + s[:4]))
    
    # Swap pairs
    ops.append(('swap_pairs', lambda s: ''.join(s[i+1]+s[i] for i in range(0,8,2))))
    
    # Try single operations
    for name, op in ops:
        if all(op(inp) == out for inp, out in examples):
            return op(query)
    
    # Try 2-operation compositions with a focused set
    # Only compose: rotations, reverse, not, swap_nibbles, swap_pairs with XOR
    structural_ops = [o for o in ops if not o[0].startswith('xor_')]
    xor_ops = [o for o in ops if o[0].startswith('xor_')]
    
    # structural then XOR
    for name1, op1 in structural_ops:
        for name2, op2 in xor_ops:
            if all(op2(op1(inp)) == out for inp, out in examples):
                return op2(op1(query))
    
    # XOR then structural
    for name1, op1 in xor_ops:
        for name2, op2 in structural_ops:
            if all(op2(op1(inp)) == out for inp, out in examples):
                return op2(op1(query))
    
    # structural then structural
    for name1, op1 in structural_ops:
        for name2, op2 in structural_ops:
            if all(op2(op1(inp)) == out for inp, out in examples):
                return op2(op1(query))
    
    # XOR then XOR (= XOR with xor of both constants, already covered by single XOR)
    # Skip
    
    return None

# Test on small sample first (this is the slow one)
bit_subset = train_df[train_df['category'] == 'bit_manipulation'].head(20).copy()
bit_subset['predicted'] = bit_subset['prompt'].apply(solve_bit_manipulation)
bit_subset['correct'] = bit_subset['predicted'] == bit_subset['answer'].astype(str)
bit_subset['solved'] = bit_subset['predicted'].notna()
print(f"Bit Manipulation (first 20):")
print(f"  Solved: {bit_subset['solved'].sum()}/20")
print(f"  Correct: {bit_subset['correct'].sum()}/20")

# Show unsolved
unsolved = bit_subset[~bit_subset['solved']]
if len(unsolved) > 0:
    print(f"\n  Unsolved examples:")
    for _, row in unsolved.head(3).iterrows():
        examples = re.findall(r'([01]{8}) -> ([01]{8})', row['prompt'])
        print(f"    Expected: {row['answer']}")
        print(f"    Examples: {examples[:3]}...")

Bit Manipulation (first 20):
  Solved: 2/20
  Correct: 2/20

  Unsolved examples:
    Expected: 10010111
    Examples: [('01010001', '11011101'), ('00001001', '01101101'), ('00010101', '01010101')]...
    Expected: 01000011
    Examples: [('10001110', '00100110'), ('10011001', '01000100'), ('01100100', '00010001')]...
    Expected: 11111111
    Examples: [('11101001', '01111101'), ('11010100', '11111110'), ('00110011', '00000111')]...


In [11]:
# Investigate bit manipulation patterns more deeply
# Some might be 3-op compositions or use bit-level permutations

def analyze_bit_pattern(prompt):
    """Analyze a bit manipulation puzzle to understand the transformation."""
    examples = re.findall(r'([01]{8}) -> ([01]{8})', prompt)
    if not examples:
        return None
    
    # Check if it's a bit permutation (each output bit is a fixed input bit or its complement)
    n = len(examples)
    inputs = np.array([[int(b) for b in inp] for inp, _ in examples])  # n x 8
    outputs = np.array([[int(b) for b in out] for _, out in examples])  # n x 8
    
    # For each output bit position, check which input bit position it matches
    permutation = []
    for out_pos in range(8):
        found = False
        for in_pos in range(8):
            # Direct match
            if np.array_equal(outputs[:, out_pos], inputs[:, in_pos]):
                permutation.append((in_pos, False))  # not inverted
                found = True
                break
            # Inverted match
            if np.array_equal(outputs[:, out_pos], 1 - inputs[:, in_pos]):
                permutation.append((in_pos, True))  # inverted
                found = True
                break
        if not found:
            permutation.append(None)  # can't explain this bit
    
    return permutation

# Test this analysis
bit_df = train_df[train_df['category'] == 'bit_manipulation'].head(50)
perm_solved = 0
for _, row in bit_df.iterrows():
    perm = analyze_bit_pattern(row['prompt'])
    if perm and all(p is not None for p in perm):
        perm_solved += 1

print(f"Bit-permutation model explains: {perm_solved}/50 puzzles")

Bit-permutation model explains: 7/50 puzzles


In [12]:
def solve_bit_permutation(prompt):
    """Solve bit manipulation as a bit permutation with optional inversion."""
    examples = re.findall(r'([01]{8}) -> ([01]{8})', prompt)
    query_match = re.search(r'determine the output for:\s*([01]{8})', prompt)
    if not examples or not query_match:
        return None

    query = query_match.group(1)
    n = len(examples)
    inputs = np.array([[int(b) for b in inp] for inp, _ in examples])
    outputs = np.array([[int(b) for b in out] for _, out in examples])

    result = []
    for out_pos in range(8):
        found = False
        for in_pos in range(8):
            if np.array_equal(outputs[:, out_pos], inputs[:, in_pos]):
                result.append(query[in_pos])
                found = True
                break
            if np.array_equal(outputs[:, out_pos], 1 - inputs[:, in_pos]):
                result.append(str(1 - int(query[in_pos])))
                found = True
                break
        if not found:
            return None  # Can't solve with bit permutation

    return ''.join(result)


def solve_bit_combined(prompt):
    """Try bit permutation first (fast), then fall back to operation search."""
    result = solve_bit_permutation(prompt)
    if result is not None:
        return result
    return solve_bit_manipulation(prompt)


# Test on larger sample
bit_subset = train_df[train_df['category'] == 'bit_manipulation'].head(100).copy()
bit_subset['predicted'] = bit_subset['prompt'].apply(solve_bit_combined)
bit_subset['correct'] = bit_subset['predicted'] == bit_subset['answer'].astype(str)
bit_subset['solved'] = bit_subset['predicted'].notna()
print(f"Bit Manipulation Combined (first 100):")
print(f"  Solved: {bit_subset['solved'].sum()}/100")
print(f"  Correct: {bit_subset['correct'].sum()}/100")

Bit Manipulation Combined (first 100):
  Solved: 12/100
  Correct: 12/100


### 2.6 Transformation Rules

In [13]:
# First, let's deeply understand the transformation rules category
tr_df = train_df[train_df['category'] == 'transformation_rules'].head(20)

for i, (_, row) in enumerate(tr_df.iterrows()):
    if i >= 10:
        break
    print(f"\n--- Puzzle {i+1} ---")
    print(f"Answer: {row['answer']}")
    # Extract the examples and query
    lines = row['prompt'].strip().split('\n')
    for line in lines:
        line = line.strip()
        if ' = ' in line and 'transformation' not in line.lower() and 'determine' not in line.lower():
            print(f"  {line}")
        elif 'determine' in line.lower():
            print(f"  QUERY: {line}")


--- Puzzle 1 ---
Answer: @&
  `!*[{ = '"[`
  \'*'> = ![@
  \'-!` = \\
  `!*\& = '@'{
  QUERY: Now, determine the result for: [[-!'

--- Puzzle 2 ---
Answer: \^?
  }`]?( = ())
  }#<)\ = #?
  ?(!&& = #@@#
  (?!@` = )#))
  QUERY: Now, determine the result for: ))!\)

--- Puzzle 3 ---
Answer: 17/
  34/44 = 1
  41/32 = 9
  34|25 = 69
  87\64 = 8853
  QUERY: Now, determine the result for: 69/52

--- Puzzle 4 ---
Answer: |@{
  `(]&: = %@#:
  &{>`% = {{
  ("'%: = {@{
  :%>&: = :"
  `('"@ = %@{
  QUERY: Now, determine the result for: {`'(&

--- Puzzle 5 ---
Answer: \([#
  %|*"| = %|"|
  \(*[^ = \([^
  (%+[@ = (%[@
  |[*([ = |[([
  [^-[( = -^
  QUERY: Now, determine the result for: \(*[#

--- Puzzle 6 ---
Answer: \:
  #]+\# = "!
  #^-{] = ]#
  \{*\! = #\^:
  QUERY: Now, determine the result for: #!-"^

--- Puzzle 7 ---
Answer: +}
  :|+>\ = {]
  |}&{> = ""@]
  @:^]] = {|
  |{&{{ = "{:@
  QUERY: Now, determine the result for: |}+@}

--- Puzzle 8 ---
Answer: 6644
  64-65 = 201
  28-68 = 861
  82/1

In [14]:
def solve_transformation_rules(prompt):
    """
    Transformation rules: parse examples as LHS = RHS equations.
    Build a character-level substitution mapping.
    These could be:
    1. Character substitution ciphers (like text encryption but with symbols)
    2. Arithmetic operations on numbers with symbol operators
    """
    lines = prompt.strip().split('\n')

    examples = []
    query = None

    for line in lines:
        line = line.strip()
        if 'determine the result for:' in line.lower():
            match = re.search(r'determine the result for:\s*(.*)', line, re.IGNORECASE)
            if match:
                query = match.group(1).strip()
        elif ' = ' in line and 'transformation rules' not in line.lower() and 'below' not in line.lower():
            parts = line.split(' = ')
            if len(parts) == 2:
                examples.append((parts[0].strip(), parts[1].strip()))

    if not examples or not query:
        return None

    # Try character-level substitution (same length LHS->RHS)
    char_map = {}
    all_same_len = all(len(lhs) == len(rhs) for lhs, rhs in examples)
    
    if all_same_len:
        conflicts = False
        for lhs, rhs in examples:
            for a, b in zip(lhs, rhs):
                if a in char_map and char_map[a] != b:
                    conflicts = True
                    break
                char_map[a] = b
            if conflicts:
                break

        if not conflicts and char_map:
            # Verify
            all_match = True
            for lhs, rhs in examples:
                predicted = ''.join(char_map.get(c, c) for c in lhs)
                if predicted != rhs:
                    all_match = False
                    break
            if all_match:
                return ''.join(char_map.get(c, c) for c in query)

    # If not same-length char substitution, might be arithmetic
    # Try to detect numeric patterns
    # Check if LHS contains operators and numbers
    
    # Pattern: number OP number = result
    # Try various operator interpretations
    numeric_pattern = re.compile(r'^(\d+)([^\d])(\d+)$')
    
    parsed_examples = []
    for lhs, rhs in examples:
        m = numeric_pattern.match(lhs)
        if m:
            parsed_examples.append((int(m.group(1)), m.group(2), int(m.group(3)), rhs))
    
    if len(parsed_examples) == len(examples) and parsed_examples:
        # Try to figure out what each operator does
        ops_seen = defaultdict(list)
        for a, op, b, result in parsed_examples:
            ops_seen[op].append((a, b, result))
        
        # For each operator, try common arithmetic operations
        op_funcs = {}
        for op, cases in ops_seen.items():
            for func_name, func in [
                ('add', lambda a, b: str(a + b)),
                ('sub', lambda a, b: str(a - b)),
                ('mul', lambda a, b: str(a * b)),
                ('concat', lambda a, b: str(a) + str(b)),
                ('div', lambda a, b: str(a // b) if b != 0 else None),
                ('mod', lambda a, b: str(a % b) if b != 0 else None),
                ('sub_rev', lambda a, b: str(b - a)),
                ('div_rev', lambda a, b: str(b // a) if a != 0 else None),
                ('xor', lambda a, b: str(a ^ b)),
                ('and', lambda a, b: str(a & b)),
                ('or', lambda a, b: str(a | b)),
                ('sub_abs', lambda a, b: str(abs(a - b))),
                ('concat_sum', lambda a, b: str(a + b) if len(str(a+b)) == len(str(a))+len(str(b)) else None),
            ]:
                if all(func(a, b) == result for a, b, result in cases):
                    op_funcs[op] = func
                    break
        
        if op_funcs:
            # Apply to query
            m = numeric_pattern.match(query)
            if m:
                a, op, b = int(m.group(1)), m.group(2), int(m.group(3))
                if op in op_funcs:
                    return op_funcs[op](a, b)

    return None


# Validate
subset = train_df[train_df['category'] == 'transformation_rules'].head(100).copy()
subset['predicted'] = subset['prompt'].apply(solve_transformation_rules)
subset['correct'] = subset['predicted'] == subset['answer'].astype(str)
subset['solved'] = subset['predicted'].notna()
print(f"Transformation Rules (first 100):")
print(f"  Solved: {subset['solved'].sum()}/100")
print(f"  Correct: {subset['correct'].sum()}/100")

# Show some unsolved/wrong
unsolved = subset[~subset['solved']].head(5)
for _, row in unsolved.iterrows():
    print(f"\n  UNSOLVED - Expected: {row['answer']}")
    lines = row['prompt'].strip().split('\n')
    for line in lines:
        line = line.strip()
        if ' = ' in line and 'transformation' not in line.lower() and 'below' not in line.lower():
            print(f"    {line}")
        elif 'determine' in line.lower():
            print(f"    QUERY: {line}")

Transformation Rules (first 100):
  Solved: 12/100
  Correct: 7/100

  UNSOLVED - Expected: @&
    `!*[{ = '"[`
    \'*'> = ![@
    \'-!` = \\
    `!*\& = '@'{
    QUERY: Now, determine the result for: [[-!'

  UNSOLVED - Expected: \^?
    }`]?( = ())
    }#<)\ = #?
    ?(!&& = #@@#
    (?!@` = )#))
    QUERY: Now, determine the result for: ))!\)

  UNSOLVED - Expected: 17/
    34/44 = 1
    41/32 = 9
    34|25 = 69
    87\64 = 8853
    QUERY: Now, determine the result for: 69/52

  UNSOLVED - Expected: |@{
    `(]&: = %@#:
    &{>`% = {{
    ("'%: = {@{
    :%>&: = :"
    `('"@ = %@{
    QUERY: Now, determine the result for: {`'(&

  UNSOLVED - Expected: \([#
    %|*"| = %|"|
    \(*[^ = \([^
    (%+[@ = (%[@
    |[*([ = |[([
    [^-[( = -^
    QUERY: Now, determine the result for: \(*[#


## 3. Full Validation Across All Categories

In [15]:
def solve_all(prompt):
    """Master solver that dispatches to category-specific solvers."""
    category = classify_puzzle(prompt)
    solvers = {
        'numeral_system': solve_numeral_system,
        'gravitational': solve_gravitational,
        'unit_conversion': solve_unit_conversion,
        'text_encryption': solve_text_encryption,
        'bit_manipulation': solve_bit_combined,
        'transformation_rules': solve_transformation_rules,
    }
    solver = solvers.get(category)
    if solver:
        return solver(prompt)
    return None

# Run on full training set (may take a few minutes for bit_manipulation)
print("Running full validation... (bit_manipulation may take a while)")

results = []
for cat in ['numeral_system', 'gravitational', 'unit_conversion', 'text_encryption', 'transformation_rules']:
    subset = train_df[train_df['category'] == cat].copy()
    subset['predicted'] = subset['prompt'].apply(solve_all)
    subset['correct'] = subset['predicted'] == subset['answer'].astype(str)
    subset['solved'] = subset['predicted'].notna()
    
    total = len(subset)
    solved = subset['solved'].sum()
    correct = subset['correct'].sum()
    print(f"{cat}: {correct}/{total} correct ({correct/total*100:.1f}%), {solved}/{total} solved")
    results.append({'category': cat, 'total': total, 'solved': solved, 'correct': correct})

# Run bit_manipulation on a larger sample (full set may be slow)
print("\nRunning bit_manipulation on first 200...")
bit_subset = train_df[train_df['category'] == 'bit_manipulation'].head(200).copy()
bit_subset['predicted'] = bit_subset['prompt'].apply(solve_bit_combined)
bit_subset['correct'] = bit_subset['predicted'] == bit_subset['answer'].astype(str)
bit_subset['solved'] = bit_subset['predicted'].notna()
print(f"bit_manipulation (200 sample): {bit_subset['correct'].sum()}/200 correct ({bit_subset['correct'].mean()*100:.1f}%), {bit_subset['solved'].sum()}/200 solved")

Running full validation... (bit_manipulation may take a while)
numeral_system: 1576/1576 correct (100.0%), 1576/1576 solved
gravitational: 1279/1597 correct (80.1%), 1597/1597 solved
unit_conversion: 1427/1594 correct (89.5%), 1594/1594 solved
text_encryption: 605/1576 correct (38.4%), 1576/1576 solved
transformation_rules: 101/1555 correct (6.5%), 152/1555 solved

Running bit_manipulation on first 200...
bit_manipulation (200 sample): 13/200 correct (6.5%), 14/200 solved


## 4. Summary & Next Steps

### Solver Accuracy Summary

| Category | Approach | Expected Accuracy |
|---|---|---|
| Numeral System | Direct decimal→Roman conversion | ~100% |
| Gravitational | Least-squares g estimation | ~80% (rest within tolerance) |
| Unit Conversion | Least-squares scaling factor | ~85% (rest within tolerance) |
| Text Encryption | Substitution cipher mapping | ~40-60% (limited by incomplete mappings) |
| Bit Manipulation | Bit permutation + op search | TBD |
| Transformation Rules | Char substitution + arithmetic | TBD |

### Key Insight
The programmatic solvers don't need to be 100% — they serve two purposes:
1. **Understanding the puzzle structure** to generate high-quality CoT
2. **Verification** of model outputs during RL training

For the actual model, we'll generate CoT training data that teaches step-by-step reasoning.

In [ ]:
print("Next steps:")
print("1. Generate Chain-of-Thought training data for all 9,500 puzzles")
print("2. Set up LoRA fine-tuning with Unsloth")
print("3. Train and submit baseline")